In [4]:
import numpy as np
import networkx as nx
from src import triangletools


# Triangular Geometry

In [ ]:
def get_steepest_ascent_opposite_to_angle(v0, v1, v2, f0, f1, f2):
    """
    Find where the steepest-ascent ray from v0 intersects the opposite
    edge v1--v2.

    Parameters
    ----------
    v0, v1, v2 : ndarray, shape (..., d)
        Triangle vertices.

    f0, f1, f2 : ndarray, shape (...)
        Values of an affine function at the corresponding vertices.

    Returns
    -------
    r : ndarray, shape (...)
        Parameter of the intersection point on the opposite edge:

            p = (1 - r)[..., None] * v1 + r[..., None] * v2

        ``nan`` is returned when the steepest-ascent ray does not intersect
        the segment v1--v2, or when the triangle is degenerate.

    df : ndarray, shape (...)
        Magnitude of the function's gradient in the affine span of the
        triangle. ``nan`` is returned for degenerate triangles.
    """
    v0 = np.asarray(v0, dtype=float)
    v1 = np.asarray(v1, dtype=float)
    v2 = np.asarray(v2, dtype=float)

    f0, f1, f2 = np.broadcast_arrays(
        np.asarray(f0, dtype=float),
        np.asarray(f1, dtype=float),
        np.asarray(f2, dtype=float),
    )

    e1 = v1 - v0
    e2 = v2 - v0

    # Gram matrix of the triangle edge basis.
    g11 = np.einsum("...i,...i->...", e1, e1)
    g12 = np.einsum("...i,...i->...", e1, e2)
    g22 = np.einsum("...i,...i->...", e2, e2)

    y1 = f1 - f0
    y2 = f2 - f0

    # If grad(f) = a1*e1 + a2*e2, then
    #
    # [e1·e1  e1·e2] [a1] = [f1-f0]
    # [e1·e2  e2·e2] [a2]   [f2-f0]
    determinant = g11 * g22 - g12 * g12

    scale = np.maximum(g11 * g22, 1.0)
    tolerance = np.finfo(float).eps * 16.0 * scale
    nondegenerate = np.abs(determinant) > tolerance

    with np.errstate(divide="ignore", invalid="ignore"):
        a1 = (g22 * y1 - g12 * y2) / determinant
        a2 = (g11 * y2 - g12 * y1) / determinant

        # v0 + t*grad(f) reaches the opposite-edge line when
        # t*(a1 + a2) = 1. Its edge coordinate is then r = t*a2.
        coefficient_sum = a1 + a2
        r_candidate = a2 / coefficient_sum

        gradient_squared = a1 * y1 + a2 * y2
        df = np.sqrt(np.maximum(gradient_squared, 0.0))

    # The forward ray intersects the edge segment exactly when both
    # coefficients are nonnegative and their sum is positive.
    direction_tolerance = (
        np.finfo(float).eps
        * 32.0
        * np.maximum(np.maximum(np.abs(a1), np.abs(a2)), 1.0)
    )

    intersects = (
        nondegenerate
        & (coefficient_sum > direction_tolerance)
        & (a1 >= -direction_tolerance)
        & (a2 >= -direction_tolerance)
    )

    r = np.where(intersects, np.clip(r_candidate, 0.0, 1.0), np.nan)
    df = np.where(nondegenerate, df, np.nan)

    return r, df

In [ ]:
import numpy as np

v0 = np.array([0.0, 0.0])
v1 = np.array([2.0, 1.0])
v2 = np.array([0.0, 1.0])

# f(x,y) = x + y
f0 = 0.0
f1 = 0.5
f2 = 1.0

r, df = get_steepest_ascent_opposite_to_angle(v0, v1, v2, f0, f1, f2)

print("r =", r)
print("df =", df)

p = (1-r) * v1 + r * v2
print("intersection =", p)

r = nan
df = 1.0307764064044151
intersection = [nan nan]


In [101]:
def get_steepest_descent_opposite_to_angle(v0, v1, v2, f0, f1, f2):
    """
    Find where the steepest-descent ray from v0 intersects the opposite
    edge v1--v2.

    Parameters
    ----------
    v0, v1, v2 : ndarray, shape (..., d)
        Triangle vertices.

    f0, f1, f2 : ndarray, shape (...)
        Values of an affine function at the corresponding vertices.

    Returns
    -------
    r : ndarray, shape (...)
        Parameter of the intersection point on the opposite edge:

            p = (1 - r)[..., None] * v1 + r[..., None] * v2

        ``nan`` is returned when the steepest-descent ray does not intersect
        the segment v1--v2, or when the triangle is degenerate.

    df : ndarray, shape (...)
        Magnitude of the function's gradient in the affine span of the
        triangle. ``nan`` is returned for degenerate triangles.
    """
    r, df = get_steepest_ascent_opposite_to_angle(v0, v1, v2, -f0, -f1, -f2)
    df *= -1
    return r, df

In [103]:
def get_steepest_opposite_to_angle(v0, v1, v2, f0, f1, f2, ascending: bool=True):
    """
    Find where the steepest ascent or descent ray from v0 intersects the opposite
    edge v1--v2.

    Parameters
    ----------
    v0, v1, v2 : ndarray, shape (..., d)
        Triangle vertices.

    f0, f1, f2 : ndarray, shape (...)
        Values of an affine function at the corresponding vertices.

    ascending: bool
        If True seek steepest ascent, else steepest descent
    
    Returns
    -------
    r : ndarray, shape (...)
        Parameter of the intersection point on the opposite edge:

            p = (1 - r)[..., None] * v1 + r[..., None] * v2

        ``nan`` is returned when the steepest-descent ray does not intersect
        the segment v1--v2, or when the triangle is degenerate.

    df : ndarray, shape (...)
        Magnitude of the function's gradient in the affine span of the
        triangle. ``nan`` is returned for degenerate triangles.
    """
    if ascending:
        return get_steepest_ascent_opposite_to_angle(v0, v1, v2, f0, f1, f2)
    else:
        return get_steepest_descent_opposite_to_angle(v0, v1, v2, f0, f1, f2)

# Morse Smale

In [122]:
class Path:
    def __init__(self, start_index):
        self.start = int(start_index)
        self.edges = np.zeros(shape=[0, 2], dtype=int)
        self.position = np.zeros(shape=0, dtype=float)

    def add_edge_point(self, i0, i1, r):
        if hasattr(self, 'end'):
            raise ValueError('The path is already ended in the vertex')
        self.edges = np.concatenate([self.edges, [np.sort([i0, i1])]])
        self.positon = np.append(self.position, r if (i1 > i0) else 1 - r)

    def end_path(self, end_index):
        self.end = int(end_index)

    def get_cords(self, vertices):
        vertices_e0 = vertices[self.edges[:, 0]]
        vertices_e1 = vertices[self.edges[:, 1]]
        path_vertices = (1 - self.position)*vertices_e0 + self.position*vertices_e1
        if hasattr(self, 'start'):
            path_vertices = np.concatenate([[vertices[self.start]], path_vertices])
        if hasattr(self, 'end'):
            path_vertices = np.concatenate([path_vertices, [vertices[self.end]]])

In [127]:
class MorseSmale:
    def __init__(self, faces, values, vertices=None):
        r"""
        Initialize the 2-dimensional simplicilial complex with N vertices and M 2-faces to quadrangulate

        Parameters:
        -----------
        faces: array shape (M, 3)
            The list of 2-faces of the simplicial complex

        values: array shape (N, )
            The filtration values of the vertices
        
        vertices: array shape (N, d) or None
            If the complex has an embeding in d-dimensional eucledean space, we can define the cords of the vertices
            If it's not given, define the complex as the Simplex embedded into d = N

        """
        self.faces = np.unique(np.sort(faces, axis=1), axis=0)
        self.values = np.array(values)
        if vertices is None:
            vertices = np.eye(len(self.values))
        self.vertices = np.array(vertices)
        if (self.vertices.shape[0] != self.values.shape[0]) or (self.vertices.ndim != 2):
            raise ValueError(f'Expected vertices length ({self.values.shape[0]}, d)')
        
        self.n_vertices = self.values.shape[0]
        self.n_edges = np.unique(np.sort(np.concatenate(self.faces[:, [[0, 1], [0, 2], [1, 2]]]), axis=1), axis=0).shape[0]
        self.n_faces = self.faces.shape[0]


    def define_critical_points(self):
        """
        Define critical points

        Attributes:
        -----------
        mins: list
            Indicises of local minimum vertices

        maxs: list
            Indicises of local maximum vertices
            
        saddles: list
            Indicises of saddle vertices
        """
        if (hasattr(self, 'mins') and hasattr(self, 'maxs') and hasattr(self, 'saddles')):
            return None
        
        self.mins = []
        self.maxs = []
        self.saddles = []
        for node in range(self.n_vertices):
            neighborhood_faces = self.faces[(self.faces == node).any(axis=1)]
            neighborhood_edges = neighborhood_faces[neighborhood_faces != node].reshape(-1, 2)
            neighborhood_nodes = np.unique(neighborhood_edges)
            neighborhood_grads = self.gradient(node, neighborhood_nodes)
            if (neighborhood_grads > 0).all():
                self.mins.append(node)
            elif (neighborhood_grads < 0).all():
                self.maxs.append(node)
            else:
                graph_neighborhood = nx.Graph()
                graph_neighborhood.add_nodes_from(neighborhood_nodes)
                graph_neighborhood.add_edges_from(neighborhood_edges)
                graph_lower_neighborhood = graph_neighborhood.subgraph(neighborhood_nodes[neighborhood_grads < 0])
                graph_higher_neighborhood = graph_neighborhood.subgraph(neighborhood_nodes[neighborhood_grads > 0])
                regular = nx.is_connected(graph_lower_neighborhood) and nx.is_connected(graph_higher_neighborhood)
                if not regular:
                    self.saddles.append(node)


    def iterate_saddles_and_increasing_directions(self):
        """
        Iterate the origins (first 2 nodes) of increasing path

        Yields:
        -------
        saddle: int
            The index of the 1st node in the increasing path
            This is always a saddle

        next_node: int
            The index of the 2nd node in the increasing path
        """
        if not hasattr(self, 'saddles'):
            self.define_critical_points()
        for saddle in self.saddles:
            neighborhood_graph = triangletools.get_neighborhood_graph(self.faces, saddle, with_center=False)
            neighbors = np.array(list(neighborhood_graph.nodes()))
            neighbors_gradients = self.gradient(saddle, neighbors)
            graph_higher_neighborhood = neighborhood_graph.subgraph(neighbors[neighbors_gradients > 0])
            for component in nx.connected_components(graph_higher_neighborhood):
                next_node = list(component)[self.gradient(saddle, list(component)).argmax()]
                yield (saddle, next_node)


    def iterate_saddles_and_decreasing_directions(self):
        """
        Iterate the origins (first 2 nodes) of decreasing path

        Yields:
        -------
        saddle: int
            The index of the 1st node in the decreasing path
            This is always a saddle

        next_node: int
            The index of the 2nd node in the decreasing path
        """
        if not hasattr(self, 'saddles'):
            self.define_critical_points()
        
        for saddle in self.saddles:
            neighborhood_graph = triangletools.get_neighborhood_graph(self.faces, saddle, with_center=False)
            neighbors = np.array(list(neighborhood_graph.nodes()))
            neighbors_gradients = self.gradient(saddle, neighbors)
            graph_lower_neighborhood = neighborhood_graph.subgraph(neighbors[neighbors_gradients < 0])
            for component in nx.connected_components(graph_lower_neighborhood):
                next_node = list(component)[self.gradient(saddle, list(component)).argmin()]
                yield (saddle, next_node)



    def continue_path_from_edge(self, i0, i1, r, increasing=True):
        """
        """
        v0 = (1 - r)*self.vertices[i0] + r*self.vertices[i1]
        f0 = (1 - r)*self.values[i0] + r*self.values[i1]
        v10 = self.vertices[i0]
        f10 = self.values[i0]
        v11 = self.vertices[i1]
        f11 = self.values[i1]
        v2_indices = self.faces[(self.faces == i0).any(axis=1) & (self.faces == i1).any(axis=1)]
        v2_indices = v2_indices[(v2_indices != i0) & (v2_indices != i1)]
        v2 = self.vertices[v2_indices]
        f2 = self.values[v2_indices]
        r0, df0 = get_steepest_opposite_to_angle(v0, v10, v2, f0, f10, f2, ascending=increasing)
        r1, df1 = get_steepest_opposite_to_angle(v0, v11, v2, f0, f11, f2, ascending=increasing)
        df0 = abs(df0)
        df1 = abs(df1)
        df0[np.isnan(df0)] = -1
        df1[np.isnan(df1)] = -1
        argmax0 = np.argmax(df0)
        argmax1 = np.argmax(df1)
        if df0[argmax0] > df1[argmax1]:
            next_i0, next_i1 = np.sort([v2_indices[argmax0], i0])
            next_r = r0[argmax0]
        else:
            next_i0, next_i1 = np.sort([v2_indices[argmax1], i1])
            next_r = r1[argmax1]
        return next_i0, next_i1, next_r

    def continue_path_from_vertex(self, i0, i1=None, increasing=True):
        """
        """
        if i1 is None:
            neighbour_indices = self.faces[(self.faces == i0).any(axis=1)]
            neighbour_indices = neighbour_indices[neighbour_indices != i0]
            if increasing:
                neighbour_indices = neighbour_indices[self.values[neighbour_indices] > self.values[i0]]
            else:
                neighbour_indices = neighbour_indices[self.values[neighbour_indices] < self.values[i0]]
            i1 = neighbour_indices[np.argmax(abs(self.values[neighbour_indices] - self.values[i0]))]


        i2s = self.faces[(self.faces == i0).any(axis=1) & (self.faces == i1).any(axis=1)]
        i2s = i2s[(i2s != i0) & (i2s != i1)]
        v0, v1 = self.vertices[[i0, i1]]
        v2s = self.vertices[i2s]
        f0, f1 = self.values[[i0, i1]]
        f2s = self.values[i2s]

        r, df = get_steepest_opposite_to_angle(v0, v1, v2s, f0, f1, f2s, ascending=increasing)
        df = abs(df)
        df[np.isnan(df)] = -1
        next_i0 = i1
        next_i1 = i2s[np.argmax(df)]
        next_r = r[np.argmax(df)]
        return next_i0, next_i1, next_r



    def get_path(self, saddle, next_node):
        """
        """
        increasing = self.values[next_node] > self.values[saddle]

        path = Path(saddle)
        next_i0, next_i1, next_r = self.continue_path_from_vertex(saddle, next_node, increasing)
        path.add_edge_point(next_i0, next_i1, next_r)

        if increasing:
            targets = self.maxs
        else:
            targets = self.mins
        while not ((next_i0 in targets) or (next_i1 in targets)):
            next_i0, next_i1, next_r = self.continue_path_from_edge(next_i0, next_i1, next_r, increasing)
            path.add_edge_point(next_i0, next_i1, next_r)

        if (next_i0 in targets) and not (next_i1 in targets):
            path.end_path(next_i0)
        elif not (next_i0 in targets) and (next_i1 in targets):
            path.end_path(next_i1)
        else:
            raise ValueError('There are 2 neghboring ending points')


        

        




        

In [128]:
path = Path(0)
path.add_edge_point(1, 2, 3)

path.edges[-1]

array([1, 2])

In [129]:
a = np.array([-3, np.nan, 0, 1, 2])
a

array([-3., nan,  0.,  1.,  2.])

In [130]:
np.where(a > 0)

(array([3, 4]),)

In [131]:
np.argmax(a)

np.int64(1)